In [ ]:
import json

import networkx as nx
from tqdm import tqdm
from nltk.corpus import wordnet as wn
import nltk
import numpy as np
from node2vec import Node2Vec

### Node2Vec

#### Weighted Graph


In [ ]:
weighted_graph = nx.read_gexf('data/graph_weighted_with_probs.gexf')

##### Generate Random Walks

In [ ]:
def random_walk(start_node, graph, q, p, walk_length):
    walk = []
    curr = start_node
    prev = start_node
    while len(walk) < walk_length:
        walk.append(curr)
        out_edges = [[edge[1], edge[2]['weight']] for edge in graph.out_edges(curr, data=True) if ((", 'meta'" not in edge[1]) \
                                                                                                    or (curr is start_node))
                    ]
        in_edges = [edge[0] for edge in graph.in_edges(curr) if edge[0] in walk]

        neighbors = [edge[0] for edge in out_edges] + in_edges

        if len(neighbors) == 0:
            return walk
            
        # NOTE: I know there is a bug here with `1/p` instead of `p/len(in_edges)` or even
        # `p` being the probability of returning strictly to `prev`. Doesn't quite align with the method described in the paper,
        # but this is the code used to generate the plots in the paper. 
        raw_probs = [edge[1] / q for edge in out_edges] + [1/p for edge in in_edges]        
        probs = np.array(raw_probs) / sum(raw_probs)
        _next = neighbors[np.random.choice(len(neighbors), p=probs)]
        prev = curr
        curr = _next
        
    return walk


In [ ]:
len_random_walk = 10
walks_per_node = 30
dim_embeddings = 200
p = 20
q = 0.25

In [ ]:
walks_weighted = []
num_nodes = len(weighted_graph.nodes())
loop = tqdm(total=num_nodes*walks_per_node)
k = 0
for node in weighted_graph.nodes():
    for i in range(walks_per_node):
        walk = []
        while len(walk) == 0:
            walk = random_walk(node, weighted_graph, q, p, len_random_walk)
        walks_weighted.append(walk)

    k += 1
    if k % 5 == 0:
        loop.update(5*walks_per_node)
loop.close()
        

In [ ]:
walks = [w for w in walks_weighted if len(w) == len_random_walk]

In [ ]:
out_file = open('data/random_walks.json', 'w')
json.dump(walks, out_file, indent=6)
out_file.close()

##### Word2vec

In [ ]:
from gensim.models import Word2Vec

In [ ]:
with open('data/random_walks.json', 'r') as file:
    node_contexts = json.load(file)

In [ ]:
class Contexts():
    def __init__(self, data):
        self.data = data

    def __iter__(self):
        for context in tqdm(self.data, desc="Training..."):
            yield context

In [ ]:
node_contexts = Contexts(node_contexts)
model = Word2Vec(node_contexts, vector_size=200, window=5, min_count=1, sg=1, workers=4, epochs=8)

In [ ]:
model.save("data/embeddings5.model")

#### Unweighted Graph

In [ ]:
graph_original = nx.read_gexf('data/pos_tagged_graph_original.gexf')

In [ ]:
node2vec1 = Node2Vec(graph_original, dimensions=64, walk_length=30, num_walks=10, workers=4,  p=1, q=.25)

In [ ]:
model1 = node2vec1.fit(window=10, min_count=1)

In [ ]:
model1.save('data/unweighted_graph_embeddings.model')

#### Exploratory

In [ ]:
wn.synsets('king')[0].definition()

In [ ]:
wn.synsets('easygoing')

In [ ]:
nltk.pos_tag(['easygoing'])